In [2]:
from pathlib import Path

WORK_DIR = Path.cwd()
KNOWLEDGE_DIR = WORK_DIR / "knowledge_base"
DATA_DIR = WORK_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = DATA_DIR / "milvus_qwen_demo.db"

if not KNOWLEDGE_DIR.exists():
    raise FileNotFoundError(
        f"没有找到 {KNOWLEDGE_DIR}。请把当前工作目录切换到 Notebook 所在目录。"
    )

docx_files = sorted(KNOWLEDGE_DIR.glob("*.docx"))
if not docx_files:
    raise FileNotFoundError(f"{KNOWLEDGE_DIR} 中没有 DOCX 文件")

print("工作目录：", WORK_DIR)
print("数据库文件：", DB_PATH)
print(f"发现 {len(docx_files)} 份 DOCX：")
for path in docx_files:
    print("-", path.name)


工作目录： /Users/jackhu/src_code/ai-infra-agent/agent-tutorial/milvus-demo
数据库文件： /Users/jackhu/src_code/ai-infra-agent/agent-tutorial/milvus-demo/data/milvus_qwen_demo.db
发现 6 份 DOCX：
- 01_员工休假制度.docx
- 02_费用报销制度.docx
- 03_远程办公制度.docx
- 04_办公区域安全制度.docx
- 05_线上故障响应制度.docx
- 06_员工培训制度.docx


In [3]:
from collections import defaultdict

from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import MetadataMode
from llama_index.readers.file import DocxReader

documents = SimpleDirectoryReader(
    input_dir=str(KNOWLEDGE_DIR),
    required_exts=[".docx"],
    file_extractor={".docx": DocxReader()},
    recursive=False,
    raise_on_error=True,
).load_data()

splitter = SentenceSplitter(
    chunk_size=350,
    chunk_overlap=50,
)
nodes = splitter.get_nodes_from_documents(documents)

chunk_counter = defaultdict(int)
chunks = []

for node in nodes:
    source_file = node.metadata.get("file_name")
    if not source_file and node.metadata.get("file_path"):
        source_file = Path(node.metadata["file_path"]).name

    text = node.get_content(metadata_mode=MetadataMode.NONE).strip()
    if not text:
        continue

    chunk_index = chunk_counter[source_file]
    chunk_counter[source_file] += 1

    chunks.append(
        {
            "text": text,
            "source_file": source_file,
            "chunk_index": chunk_index,
        }
    )

print(f"Document 数量：{len(documents)}")
print(f"Chunk 数量：{len(chunks)}")
print("第一条 Chunk：")
print(chunks[0])


Document 数量：6
Chunk 数量：12
第一条 Chunk：
{'text': '示例科技有限公司｜内部制度文件\n\n员工休假制度\n\n企业内部管理制度\n\n归口部门：人力资源部\n\n适用范围：中国大陆地区正式员工\n\n生效日期：2026年1月1日\n\n文件版本：V1.0\n\n\n\n一、制度目的\n\n规范员工带薪年假的申请、审批与结转管理，保障员工休息权益并维持团队工作的连续性。\n\n二、制度内容\n\n第一条 年假额度\n\n正式员工每个自然年度享有10个工作日的带薪年假。新入职员工当年度的年假额度按照实际在职月份折算。\n\n年假以工作日为计算单位，国家法定节假日和周末不计入已使用的年假天数。\n\n第二条 申请与审批\n\n员工应至少提前3个工作日在OA系统提交年假申请，并由直属主管审批。', 'source_file': '01_员工休假制度.docx', 'chunk_index': 0}


In [6]:
import dashscope
import os

EMBEDDING_MODEL = "qwen3.7-text-embedding"
dashscope.api_key = os.environ["DASHSCOPE_API_KEY"]


def embed_texts(texts: list[str], batch_size: int = 8) -> list[list[float]]:
    # 批量调用模型，并保持输出顺序与输入一致。
    if not texts:
        return []

    all_vectors = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        response = dashscope.TextEmbedding.call(
            model=EMBEDDING_MODEL,
            input=batch,
        )

        if response.status_code != 200:
            raise RuntimeError(
                f"Embedding 请求失败：{response.code} - {response.message}"
            )

        ordered_items = sorted(
            response.output["embeddings"],
            key=lambda item: item["text_index"],
        )
        all_vectors.extend(item["embedding"] for item in ordered_items)

    if len(all_vectors) != len(texts):
        raise RuntimeError("Embedding 数量与输入文本数量不一致")

    return all_vectors


dimension_probe = embed_texts(["这是用于检测向量维度的文本"])[0]
EMBEDDING_DIM = len(dimension_probe)

print("Embedding 模型：", EMBEDDING_MODEL)
print("向量维度：", EMBEDDING_DIM)
print("前 5 个值：", dimension_probe[:5])


Embedding 模型： qwen3.7-text-embedding
向量维度： 1024
前 5 个值： [0.027331871911883354, 0.03176603093743324, -0.0005497268284671009, -0.0062332660891115665, -0.020389866083860397]


In [7]:
chunk_texts = [item["text"] for item in chunks]
chunk_vectors = embed_texts(chunk_texts, batch_size=8)

assert len(chunk_vectors) == len(chunks)
assert all(len(vector) == EMBEDDING_DIM for vector in chunk_vectors)

print(f"已经为 {len(chunk_vectors)} 个 Chunk 生成向量")


已经为 12 个 Chunk 生成向量


In [8]:
from pymilvus import MilvusClient

client = MilvusClient(uri=str(DB_PATH))

print("当前 Collections：", client.list_collections())
print("数据库文件是否存在：", DB_PATH.exists())


2026-08-29 17:00:12,128 - INFO - Loading faiss.
2026-08-29 17:00:12,203 - INFO - Successfully loaded faiss.
2026-08-29 17:00:12,267 - INFO - MilvusLite server started for /Users/jackhu/src_code/ai-infra-agent/agent-tutorial/milvus-demo/data/milvus_qwen_demo.db on port 53656


当前 Collections： []
数据库文件是否存在： True


I0829 17:01:32.389666 3186206 chttp2_transport.cc:1393] ipv4:127.0.0.1:53656: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E0829 17:01:32.389824 3186206 chttp2_transport.cc:1425] ipv4:127.0.0.1:53656: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


In [12]:
COLLECTION_NAME = "enterprise_kb_demo"

if client.has_collection(COLLECTION_NAME):
    client.drop_collection(COLLECTION_NAME)
    print("已删除旧的 Collection：", COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=EMBEDDING_DIM,
    primary_field_name="id",
    vector_field_name="vector",
    metric_type="COSINE",
    auto_id=False,
    enable_dynamic_field=True,
)

print("创建完成：")
print(client.describe_collection(COLLECTION_NAME))


2026-08-29 17:07:15,971 - ERROR - Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/opt/anaconda3/envs/agent-base/lib/python3.13/site-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
  File "/opt/anaconda3/envs/agent-base/lib/python3.13/site-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1264, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!


已删除旧的 Collection： enterprise_kb_demo
创建完成：
{'collection_name': 'enterprise_kb_demo', 'auto_id': False, 'num_shards': 1, 'description': '', 'fields': [{'field_id': 0, 'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'params': {}, 'is_primary': True}, {'field_id': 0, 'name': 'vector', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1024}}], 'functions': [], 'aliases': [], 'collection_id': 0, 'consistency_level': 0, 'consistency_level_name': 'Strong', 'properties': {}, 'num_partitions': 1, 'enable_dynamic_field': True, 'enable_namespace': False, 'schema_version': 0}


I0829 17:09:56.736380 3186210 chttp2_transport.cc:1393] ipv4:127.0.0.1:53656: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E0829 17:09:56.736489 3186210 chttp2_transport.cc:1425] ipv4:127.0.0.1:53656: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 40000ms


In [13]:
entities = [
    {
        "id": index,
        "vector": chunk_vectors[index],
        "text": chunk["text"],
        "source_file": chunk["source_file"],
        "chunk_index": chunk["chunk_index"],
    }
    for index, chunk in enumerate(chunks)
]

insert_result = client.insert(
    collection_name=COLLECTION_NAME,
    data=entities,
)

print("写入结果：", insert_result)
print("Collection 统计：", client.get_collection_stats(COLLECTION_NAME))


写入结果： {'insert_count': 12, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]}
Collection 统计： {'row_count': 12}


In [14]:
import pandas as pd

def semantic_search(
    query: str,
    limit: int = 3,
    filter_expression: str = "",
) -> pd.DataFrame:
    query_vector = embed_texts([query])[0]

    search_kwargs = {
        "collection_name": COLLECTION_NAME,
        "data": [query_vector],
        "limit": limit,
        "output_fields": ["text", "source_file", "chunk_index"],
    }
    if filter_expression:
        search_kwargs["filter"] = filter_expression

    results = client.search(**search_kwargs)

    rows = []
    for rank, hit in enumerate(results[0], start=1):
        entity = hit.get("entity", {})
        rows.append(
            {
                "rank": rank,
                "id": hit["id"],
                "score": hit["distance"],
                "source_file": entity.get("source_file"),
                "chunk_index": entity.get("chunk_index"),
                "text": entity.get("text"),
            }
        )

    return pd.DataFrame(rows)


## 10. 执行第一次语义搜索

用户问题和文档不需要包含完全相同的词。Embedding 模型负责表达语义，Milvus 负责在向量空间中寻找最近邻。


In [15]:
question = "我的门禁卡找不到了，需要多久之内上报？"
search_df = semantic_search(question, limit=3)

print("问题：", question)
display(search_df)


问题： 我的门禁卡找不到了，需要多久之内上报？


,rank,id,score,source_file,chunk_index,text
0,1,7,0.739517,04_办公区域安全制度.docx,1,一、制度目的\n\n规范办公区域身份凭证的使用和遗失处理，降低未授权访问风险。\n\n二、制...
1,2,8,0.489230,05_线上故障响应制度.docx,0,示例科技有限公司｜内部制度文件\n\n线上故障响应制度\n\n企业内部管理制度\n\n归口部...
2,3,9,0.455305,05_线上故障响应制度.docx,1,一、制度目的\n\n建立统一的线上故障分级和响应要求，缩短核心业务中断时间并形成可复盘记录。...


In [16]:
filtered_df = semantic_search(
    query="一笔6000元的费用需要谁审批？",
    limit=3,
    filter_expression='source_file == "02_费用报销制度.docx"',
)

display(filtered_df)


,rank,id,score,source_file,chunk_index,text
0,1,3,0.599938,02_费用报销制度.docx,1,一、制度目的\n\n统一业务费用的申请、凭证和审批要求，确保公司费用真实、合规并可追溯。\n...
1,2,2,0.531384,02_费用报销制度.docx,0,示例科技有限公司｜内部制度文件\n\n费用报销制度\n\n企业内部管理制度\n\n归口部门：...


I0829 17:18:20.959970 3186205 chttp2_transport.cc:1393] ipv4:127.0.0.1:53656: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E0829 17:18:20.960074 3186205 chttp2_transport.cc:1425] ipv4:127.0.0.1:53656: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 80000ms


In [17]:
query_rows = client.query(
    collection_name=COLLECTION_NAME,
    filter='source_file == "01_员工休假制度.docx"',
    output_fields=["id", "text", "source_file", "chunk_index"],
)

print("Query 结果数量：", len(query_rows))
display(pd.DataFrame(query_rows))

first_id = entities[0]["id"]
get_rows = client.get(
    collection_name=COLLECTION_NAME,
    ids=[first_id],
    output_fields=["id", "text", "source_file", "chunk_index"],
)

print("Get 结果：")
display(pd.DataFrame(get_rows))


Query 结果数量： 2


,id,text,source_file,chunk_index
0,0,示例科技有限公司｜内部制度文件\n\n员工休假制度\n\n企业内部管理制度\n\n归口部门：...,01_员工休假制度.docx,0
1,1,第二条 申请与审批\n\n员工应至少提前3个工作日在OA系统提交年假申请，并由直属主管审批。...,01_员工休假制度.docx,1


Get 结果：


,id,text,source_file,chunk_index
0,0,示例科技有限公司｜内部制度文件\n\n员工休假制度\n\n企业内部管理制度\n\n归口部门：...,01_员工休假制度.docx,0


In [18]:
TEMP_ID = 9999
temp_text_v1 = "临时制度：技术分享会安排在每周五下午。"

client.upsert(
    collection_name=COLLECTION_NAME,
    data=[
        {
            "id": TEMP_ID,
            "vector": embed_texts([temp_text_v1])[0],
            "text": temp_text_v1,
            "source_file": "临时制度.docx",
            "chunk_index": 0,
        }
    ],
)
print("首次 Upsert：", client.get(COLLECTION_NAME, ids=[TEMP_ID]))

temp_text_v2 = "临时制度：技术分享会调整为每周四下午。"
client.upsert(
    collection_name=COLLECTION_NAME,
    data=[
        {
            "id": TEMP_ID,
            "vector": embed_texts([temp_text_v2])[0],
            "text": temp_text_v2,
            "source_file": "临时制度.docx",
            "chunk_index": 0,
        }
    ],
)
print("更新后：", client.get(COLLECTION_NAME, ids=[TEMP_ID]))

delete_result = client.delete(
    collection_name=COLLECTION_NAME,
    ids=[TEMP_ID],
)
print("删除结果：", delete_result)
print("删除后：", client.get(COLLECTION_NAME, ids=[TEMP_ID]))


首次 Upsert： data: ["{'id': 9999, 'vector': [0.017225367948412895, 0.029341518878936768, 0.02880626730620861, -0.01919606700539589, -0.0064047714695334435, 0.036883700639009476, -0.025327131152153015, -0.06204052269458771, -0.012675730511546135, 0.017553819343447685, -0.002603268949314952, 0.035326603800058365, -0.006568996701389551, -0.016288679093122482, -0.004741234239190817, -0.007803724613040686, -0.01302850991487503, -0.02279685065150261, -0.015765592455863953, 0.06014281138777733, 0.03415878117084503, -0.023052312433719635, 0.024487759917974472, 0.03544825315475464, 0.05007035285234451, 0.07839002460241318, 0.018137728795409203, -0.021239755675196648, 0.04111705347895622, -0.008545777760446072, 0.054984934628009796, -0.038684092462062836, 0.02103295363485813, -0.017091555520892143, 0.010297510772943497, 0.11113768815994263, 0.03700534626841545, 0.030436350032687187, -0.04729677364230156, 0.036032162606716156, 0.020509866997599602, -0.015242504887282848, 0.04943777993321419, -0.004

In [19]:
rag_question = "P1级线上故障需要多快确认告警？"
rag_search_df = semantic_search(rag_question, limit=3)

context_blocks = []
for row in rag_search_df.to_dict("records"):
    context_blocks.append(
        f"来源：{row['source_file']}\n内容：{row['text']}"
    )

rag_context = "\n\n".join(context_blocks)

rag_prompt = (
    "请只根据参考资料回答问题；资料不足时明确说明不知道。\n\n"
    f"参考资料：\n{rag_context}\n\n"
    f"用户问题：{rag_question}"
)

print(rag_prompt)


请只根据参考资料回答问题；资料不足时明确说明不知道。

参考资料：
来源：05_线上故障响应制度.docx
内容：一、制度目的

建立统一的线上故障分级和响应要求，缩短核心业务中断时间并形成可复盘记录。

二、制度内容

第一条 P1级故障定义

P1级线上故障是影响核心业务、导致大量用户无法正常使用服务的最高优先级事件。值班人员不得擅自降低故障等级。

第二条 首次响应

值班工程师必须在10分钟内确认P1级告警，并立即建立故障处理群，通知当班负责人和相关系统负责人。

第三条 恢复与复盘

处理期间应持续记录关键操作和时间点。故障恢复后24小时内应完成初步复盘，并在三个工作日内形成正式报告。

内部资料，请妥善保管｜第 1 页

来源：05_线上故障响应制度.docx
内容：示例科技有限公司｜内部制度文件

线上故障响应制度

企业内部管理制度

归口部门：研发中心、运维团队

适用范围：参与线上系统建设、值班和支持的相关人员

生效日期：2026年1月1日

文件版本：V1.0

来源：03_远程办公制度.docx
内容：一、制度目的

明确远程办公的适用时间、申请方式和信息安全要求，保证远程协作效率。

二、制度内容

第一条 适用时间

符合条件的员工可以在每周三申请远程办公。遇到公司级会议、客户现场工作或团队集中协作安排时，应优先服从现场办公要求。

第二条 申请要求

员工应在前一个工作日下班前提交远程办公申请，经直属主管确认后方可执行。远程办公地点应具备稳定网络和安静的工作环境。

第三条 安全与在线要求

远程办公期间必须通过公司VPN访问内部系统，并在工作时间保持即时通讯在线。不得使用公共设备保存公司资料。

内部资料，请妥善保管｜第 1 页

用户问题：P1级线上故障需要多快确认告警？
